<a href="https://colab.research.google.com/github/brpetros/prompts_and_evalution_notebooks/blob/main/1_skillab_data_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Log in and functions definition

In [ ]:
import requests
from pprint import pprint
from google.colab import userdata
import os

BASE_URL = "https://skillab-tracker.csd.auth.gr/api"

def login(username, password):
    url = f"{BASE_URL}/login"

    response = requests.post(url, json={"username": username, "password": password}, verify=False)
    response.raise_for_status()
    return response.json()

os.environ['API_TOKEN']= login(userdata.get("SKILLAB_USERNAME"), userdata.get("SKILLAB_PASSWORD"))

def auth_headers(token):
    return {
        "Authorization": f"Bearer {token}",
        "Content-Type" : "application/x-www-form-urlencoded",
        "Accept": "application/json"
    }


def get_occupations(token, keywords=None, ancestors=None, keywords_logic="or"):
    url = f"{BASE_URL}/occupations"
    all_items = []
    page = 1
    page_size = 100
    payload = {}
    if keywords is not None:
        payload["keywords"] = keywords
        payload["keywords_logic"] = keywords_logic
    if ancestors is not None:
        payload["ancestors"] = ancestors

    print(payload)

    while True:
        params = {"page": page, "page_size": page_size}
        response_data = requests.post(url, params=params, data=payload, headers=auth_headers(token), verify=False).json()

        if not response_data.get('items'):
            break

        all_items.extend(response_data['items'])
        page += 1
    return {"count": len(all_items), "items": all_items}

# break an array into pieces
def chunked(iterable, size):
    for i in range(0, len(iterable), size):
        yield iterable[i:i + size]


def get_jobs(token, keywords, occupations_data, keywords_logic="or"):
    url = f"{BASE_URL}/jobs"
    jobs = {}

    page_size = 100
    CHUNK_SIZE = 25
    MAX_PAGES = 1000
    occupation_ids = list({occupation.get('id') for occupation in occupations_data.get('items')})

    for occ_chunk in chunked(occupation_ids, CHUNK_SIZE):
        page = 1

        while page <= MAX_PAGES:
            params = {"page": page, "page_size": page_size}
            payload = {"keywords": keywords, "keywords_logic": keywords_logic, "occupation_ids": occ_chunk}

            response = requests.post(
                url,
                params=params,
                data=payload,
                headers=auth_headers(token),
                verify=False
            )
            response.raise_for_status()
            data = response.json()

            items = data.get("items", [])
            if not items:
                break

            for job in items:
                jobs[job["id"]] = job

            page += 1

    return {
        "count": len(jobs),
        "items": list(jobs.values())
    }





# Searching for the agriculture related occupations using keywords



In [ ]:

keywords = ["agriculture", "agro-food",
                            "agronomy", "farming",
                            "horticulture", "livestock", "dairy",
                            "fisheries", "food processing", "crop management",
                            "sustainable agriculture", "organic farming" ]

if os.environ.get('API_TOKEN'):
    occupations_data = get_occupations(token=os.environ.get('API_TOKEN'), keywords=keywords)
    pprint(occupations_data)

else:
    print("Failed to obtain access token.")

we notice that the occupations that are returned belong to different ISCO categories, not only to ISCO category 6, which is for Skilled Agricultural, Forestry and Fishery Workers. Nevertheless, many of them are relevant. For example, we get an occupation with the label 'livestock worker', which is relevant. This belongs to ISCO category 9 (Elementary Occupations).

So our next step is to search for any category related specifically to Agriculture in the ISCO classification. We use the tables at: https://ilostat.ilo.org/methods/concepts-and-definitions/classification-occupation/ for that matter.

The categories that we found are the following:

| ISCO code | Description |
| --------- | ----------- |
| 131 | Production Managers in Agriculture, Forestry and Fisheries |
| 2131 | Biologists, Botanists, Zoologists and Related Professionals |
| 2132 | Farming, Forestry and Fisheries Advisers |
| 3142 | Agricultural Technicians |
| 6 | Skilled Agricultural, Forestry and Fishery Workers |
| 7233 | Agricultural and Industrial Machinery Mechanics and Repairers |
| 816 | Food and Related Products Machine Operators |
| 92 |  Agricultural, Forestry and Fishery Labourers |


Intentionally, we are very specific on some categories, and more broad on some others. This is because category 6 is dedicated to Agricultural Workers, so we can take all the occupations and exclude manually what is irrelevant. On the contrary, categories like 2 (Professionals) are too generic to take all the occupations, so we focus only on the more specific sub-categories.

In any case, we can manually add or exclude occupations based on the relevance of the results.


# Extraction of agriculture related occupations

We are now going to find all the occupations that have the above ISCO categories as ancestors.  




In [ ]:
ancestors = [ 'http://data.europa.eu/esco/isco/C131',


              'http://data.europa.eu/esco/isco/C3142',
              'http://data.europa.eu/esco/isco/C6',
              'http://data.europa.eu/esco/isco/C7233',
              'http://data.europa.eu/esco/isco/C816',
              'http://data.europa.eu/esco/isco/C92']

if os.environ.get('API_TOKEN'):
    occupations_data = get_occupations(token=os.environ.get('API_TOKEN'), ancestors=ancestors)
    for occupation in occupations_data['items']:
        print(occupation.get('label'))
    print(len(occupations_data['items']))
    #pprint(occupations_data)

else:
    print("Failed to obtain access token.")


# Limiting the results

We notice that using the above keywords, although the results are importantly limited, some of the job offers that are found are not significantly relevant to agriculture.

It is important that we keep only the agriculture related jobs, so that out data is accurate and worthy of analysis.


## Method

To face the above problem, we will reduce the ESCO categories used as parameters. In addition, we will perform multiple searches with different keywords each time. All the retrieved items (job items) will be stored in a dictionnary with the job id as the id.

Finally, we will perform manual removal of any job offer that is not related to agriculture.

We understand that this could potentially exclude some of the job offers, even some that are actually relevant to agriculture. Nevertheless, as thousands of job offers exist, we consider that the final statistical outcome is accurate.

We will modify our methods if needed.

## Modifications

From the ISCO table used previously, decided to exclude:

- 2131: Biologists, Botanists, Zoologists and Related Professionals

  It is too broad and specialized in a medical way, it contains skills that are not relevant or not necessary for an agriculture-related profession

- 7233: Agricultural and Industrial Machinery Mechanics and Repairers

  It concernes mechanics and agricultural staff/managers. Also, it concerns industrial machinery in general so it is also too broad.

-  816: Food and Related Products Machine Operators

Therefore, the categories we are eventually using are the following:


| ISCO code | Description |
| --------- | ----------- |
| 131 | Production Managers in Agriculture, Forestry and Fisheries |
| 2132 | Farming, Forestry and Fisheries Advisers |
| 3142 | Agricultural Technicians |
| 6 | Skilled Agricultural, Forestry and Fishery Workers |
| 92 |  Agricultural, Forestry and Fishery Labourers |

In addition, we will perform multiple searches

Getting the skills found in the job offers

In [ ]:
ancestors = [ 'http://data.europa.eu/esco/isco/C131',
            'http://data.europa.eu/esco/isco/C2132',
            'http://data.europa.eu/esco/isco/C3142',
            'http://data.europa.eu/esco/isco/C6',
            'http://data.europa.eu/esco/isco/C92']

#ancestors = ['http://data.europa.eu/esco/isco/C92']

if os.environ.get('API_TOKEN'):
    occupations_data = get_occupations(os.environ.get('API_TOKEN'), ancestors=ancestors)
    for occupation in occupations_data['items']:
        print(occupation.get('label'))
    print(len(occupations_data['items']))
    #pprint(occupations_data)

else:
    print("Failed to obtain access token.")


We now get 111 occupations, before it was 203

# Keywords based on litterature's thematic map by Bibliometrix

In [ ]:
justified_keywords = [
    "agricultural management",
    "farm management",
    "crop management"
    "sustainable agriculture",
    "food security",
    "biodiversity",
    "cropping systems",
    "land-use",
    "crop production",
    "conservation agriculture",
    "farming systems",
    "irrigation strategies",
    "manage livestock",
    "soil fertility",
    "agroecology",
    "soil organic matter",
    "water management",
    "soil health",
    "tillage",
    "no-tillage",
    "crop resilience",
    "crop yield",
    "deficit irrigation",
    "nitrogren use efficiency",
    "water productivity",
    "fertilizer use",
    "drip irrigation",
    "precision agriculture",
    "precision farming",
    "agriculture remote sensing",
    "smart agriculture",
    "crop disease",
    "pest management",
    "weed detection",
    "grain yield",
    "efficient water use",
    "animal breeding",
    "biofortification",
    "crop improvement",
    "plant breeding",
    "powdery mildew",
    "integrated pest management",
    "biological control",
    "weed management",
    "use pesticides",
    "integrated weed management",
    "crop protection"
]

print(len(justified_keywords))

In [ ]:

jobs_data = get_jobs(os.environ.get('API_TOKEN'), justified_keywords, occupations_data, "or")
# pprint(jobs_data)
for job in jobs_data['items']:
    print(job.get('title'))
print(f"successful access to jobs data - count: {jobs_data.get("count")}")





1681 returned jobs

## Filter jobs

Many of the job offers, although they contain the keywords and the occupations we searched for, they are not relevant to agriculture.

We are going to filter them based on the occupations labels that we stored above.





## filtering according to skills

We already have a set of skills related (optionally or significantly) to agriculture extracted by ESCO database.
We are going to exclude any job offer that has no skills that belong to that set

In [ ]:
import pandas as pd

try:
  df = pd.read_csv("/content/EssentialAgriculturalSkills.csv")

  relevant_esco_skills = df.set_index('skill').to_dict(orient='index')
except FileNotFoundError:
  print("File not found.")

def filter_according_to_skills(jobs_data, relevant_skills):
    accepted = {}
    rejected = {}
    for job in jobs_data.get('items'):
      for skill_id in job.get('skills'):
        if skill_id in relevant_skills:
          accepted[job['id']] = job
          break
      if job['id'] not in accepted.keys():
        rejected[job['id']] = job

    return accepted, rejected

accepted, rejected = filter_according_to_skills(jobs_data, relevant_esco_skills)
print_filter_results(accepted, rejected)

In [ ]:
a,b,c = analyze_job_result(rejected ,justified_keywords, occupations_data, job_title="Cattle keeper, in agriculture")

In [ ]:
def filter_according_to_occupations(jobs_data, occupations_data):
  accepted = {}
  rejected = {}
  occupation_ids = list({occupation.get('id') for occupation in occupations_data.get('items')})
  for job in jobs_data.values():
    relevant_occupations = []
    for occupation_id in occupation_ids:
      if occupation_id in job.get('occupations'):
        relevant_occupations.append(occupation_id)
    if len(relevant_occupations)/len(job.get('occupations')) >= 0.3:
      accepted[job['id']] = job
    else:
      rejected[job['id']] = job

  return accepted, rejected


In [ ]:
def filter_jobs(jobs, relevant_skills, occupations_data):
    accepted_by_skills , rejected_by_skills = filter_according_to_skills(jobs, relevant_skills)
    accepted_by_occupations, rejected_by_occupations = filter_according_to_occupations(accepted_by_skills, occupations_data)

    accepted = accepted_by_occupations
    rejected = rejected_by_skills | rejected_by_occupations

    return accepted, rejected


In [ ]:
accepted, rejected = filter_jobs(jobs_data, relevant_esco_skills, occupations_data)
print_filter_results(accepted, rejected)

In [ ]:
a,b,c = analyze_job_result(accepted ,justified_keywords, occupations_data, job_title="Växthusarbetare till food tech-bolag")

## extracting the skills

In [ ]:
def extract_skills (jobs):
  skills = {}
  for job in jobs.values():
    for skill_id in job.get('skills'):
      if skill_id not in skills:
        skills[skill_id] = [job['id']]
      else:
        skills[skill_id].append(job['id'])
  return skills


In [ ]:
skills = extract_skills(accepted)

pprint(skills)
print(f"total skills found: {len(skills)}")

In [ ]:
def filter_skills(skills, relevant_esco_skills):
  filtered_skills = {}
  for skill_id, job_ids in skills.items():
    if skill_id in relevant_esco_skills:
      filtered_skills[skill_id] = {'label': relevant_esco_skills.get(skill_id).get('skillLabel'), 'job_count': len(job_ids), 'job_ids': job_ids }
  return filtered_skills

skill_dict = filter_skills(skills, relevant_esco_skills)
pprint(skill_dict)
print(f"total of {len(skill_dict)} relevant skills")

# Filtering the skills

1716 skills are returned and most of them are not relevant to agriculture. So we need to filter them

We import the csv file with all the optional and essential skills related to agriculture and taking the skills column which contains the skills (the file was created using the turtle version of ESCO data and a simple sparql query to find all the skills that are optionally or essentially connected to the occupations related to agriculture - it contains 855 skills)

In [ ]:
sorted_skills = sorted(skill_dict.items(), key=lambda item: item[1]['job_count'], reverse=True)

print("--- Relevant Skills sorted by job_count (Descending) ---")
for skill_id, skill_info in sorted_skills:
    print(f"Skill ID: {skill_id}, Label: {skill_info['label']}, Job Count: {skill_info['job_count']}, Job IDs: {skill_info['job_ids']}")

# Converting and saving relevant skills as csv file

In [ ]:
def save_skills_to_csv(sorted_skills):
    skill_data = []
    for skill_id, skill_info in sorted_skills:
        skill_data.append({
            'skill_label': skill_info['label'], 'skill_id': skill_id, 'job_count': skill_info['job_count'], 'job_ids': skill_info['job_ids']
        })

    df_relevant_skills = pd.DataFrame(skill_data)

    try:
      df_relevant_skills.to_csv('relevant_skills.csv', index=False)
      print("DataFrame 'df_relevant_skills' successfully saved to 'relevant_skills.csv'")

    except Exception as e:
      print(f"Error saving DataFrame to CSV: {e}")

save_skills_to_csv(sorted_skills)

In [ ]:
import pandas as pd


df = pd.read_csv("/content/relevant_skills.csv")


df.head()


In [ ]:
import pandas as pd

def save_jobs_to_csv(sorted_skills):
    if jobs_data.get('items'):
        df_jobs_data = pd.DataFrame(jobs_data['items'])
        try:
            df_jobs_data.to_csv('all_jobs_data.csv', index=False)
            print("DataFrame 'df_jobs_data' successfully saved to 'all_jobs_data.csv'")
        except Exception as e:
            print(f"Error saving DataFrame 'df_jobs_data' to CSV: {e}")
    else:
        print("jobs_data not found or is empty.")

def save_accepted_jobs_to_csv(accepted):
    if accepted:
        df_accepted_jobs = pd.DataFrame(list(accepted.values()))
        try:
            df_accepted_jobs.to_csv('accepted_jobs.csv', index=False)
            print("DataFrame 'df_accepted_jobs' successfully saved to 'accepted_jobs.csv'")
        except Exception as e:
            print(f"Error saving DataFrame 'df_accepted_jobs' to CSV: {e}")
    else:
        print("accepted not found or is empty.")

save_jobs_to_csv(sorted_skills)
save_accepted_jobs_to_csv(accepted)

In [ ]:
import pandas as pd

df = pd.read_csv("/content/all_jobs_data.csv")
print("--- all jobs ---")
display(df.head())

df1 = pd.read_csv("/content/accepted_jobs.csv")
print("\n--- accepted jobs ---")
display(df1.head())